In [1]:
!pip install duckdb
import os
import duckdb
import pandas as pd
import pyarrow.parquet as pq
import logging
from datetime import datetime
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format = "%(asctime)s - %(message)s")
logger = logging.getLogger()

def read_parquet_as_df(parquet_path: str):
    return pq.read_table(parquet_path).to_pandas()

In [ ]:
read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\scdi_finder_output\PV.EST\Brazil.parquet")

,country_a_id,country_b_id,length,start_year_a,start_year_b,correlation
0,31,19,3,1996,1996,0.849785
1,31,19,3,1996,2002,0.665572
2,31,19,3,1996,2009,0.961957
3,31,19,3,1996,2011,0.837191
4,31,19,3,1996,2017,0.983686
...,...,...,...,...,...,...
147082,31,227,24,1998,1998,0.506067
147083,31,233,24,1998,1996,0.624471
147084,31,236,24,1998,1996,0.509874
147085,31,240,24,1998,1996,0.515810


In [ ]:
import os
import glob
import pandas as pd
import shutil
import logging


logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger()


def process_country_subfolders(base_dir):
    """
    Process each country_a_id subfolder, merge Parquets, filter, reorder, and save.
    """
    # Find all country_a_id subfolders
    subfolders = glob.glob(os.path.join(base_dir, "country_a_id=*"))
    filtered_out_scdis = 0
    
    for subfolder in subfolders:
        # Extract country_a_id from the subfolder name
        country_a_id = os.path.basename(subfolder).split('=')[1]
        logger.info(f"Processing subfolder: {subfolder} (country_a_id={country_a_id})")
        
        # Find all Parquet files in the subfolder
        parquet_files = glob.glob(os.path.join(subfolder, "*.parquet"))
        if not parquet_files:
            logger.warning(f"No Parquet files found in {subfolder}. Skipping...")
            continue
        
        # Read and merge all Parquet files
        dfs = []
        for file in parquet_files:
            df = pd.read_parquet(file)
            dfs.append(df)
        
        if not dfs:
            logger.warning(f"No data found in {subfolder}. Skipping...")
            continue
        
        merged_df = pd.concat(dfs, ignore_index=True)
        logger.info(f"Merged {len(parquet_files)} Parquet files into a DataFrame with {len(merged_df)} rows")
        
        # Filter rows where scdi_count == 1
        filtered_df = merged_df[merged_df['scdi_count'] != 1]
        logger.info(f"Filtered out {len(merged_df) - len(filtered_df)} rows with scdi_count == 1")
        filtered_out_scdis += len(merged_df) - len(filtered_df)
        
        # Reorder columns
        column_order = [
            "country_a_id",
            "country_b_id",
            "length",
            "start_year_a",
            "start_year_b",
            "average_correlation",
            "scdi_count"
        ]
        filtered_df = filtered_df.reindex(columns=column_order)
        logger.info("Reordered columns successfully")
        
        # Save the merged and filtered DataFrame to a new Parquet file
        output_filename = f"country_a_id={country_a_id}.parquet"
        output_path = os.path.join(base_dir, output_filename)
        filtered_df.to_parquet(
            output_path,
            index=False,
            compression="brotli"
        )
        logger.info(f"Saved merged and filtered data to {output_path}")
        
        # Move the new Parquet file to the parent directory (if needed)
        # and delete the original subfolder
        try:
            # Remove the original subfolder
            shutil.rmtree(subfolder)
            logger.info(f"Removed original subfolder: {subfolder}")
        except Exception as e:
            logger.error(f"Failed to remove subfolder {subfolder}: {str(e)}")
    
    logger.info(f"Filtered out a total of {filtered_out_scdis} patterns with single SCDIs.")


base_dir = r"/workspace/qgi\qgi-4.0\data\pattern_maker_output"
process_country_subfolders(base_dir)
logger.info("Processing completed")

2025-04-15 12:17:23,182 - Processing subfolder: /workspace/qgi\qgi-4.0\data\pattern_maker_output\country_a_id=1 (country_a_id=1)
2025-04-15 12:17:32,554 - Merged 488 Parquet files into a DataFrame with 14772508 rows
2025-04-15 12:17:33,023 - Filtered out 4639491 rows with scdi_count == 1
2025-04-15 12:17:33,244 - Reordered columns successfully
2025-04-15 12:17:47,667 - Saved merged and filtered data to /workspace/qgi\qgi-4.0\data\pattern_maker_output\country_a_id=1.parquet
2025-04-15 12:17:47,883 - Removed original subfolder: /workspace/qgi\qgi-4.0\data\pattern_maker_output\country_a_id=1
2025-04-15 12:17:47,884 - Processing subfolder: /workspace/qgi\qgi-4.0\data\pattern_maker_output\country_a_id=10 (country_a_id=10)
2025-04-15 12:17:57,792 - Merged 488 Parquet files into a DataFrame with 16217405 rows
2025-04-15 12:17:58,401 - Filtered out 4321330 rows with scdi_count == 1
2025-04-15 12:17:58,676 - Reordered columns successfully
2025-04-15 12:18:14,865 - Saved merged and filtered data

In [3]:
import os
import pandas as pd
import logging
import shutil
import glob

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger()

def rename_parquets_by_country_name(base_dir, countries_csv):
    """
    Rename Parquet files from "country_a_id=<id>.parquet" to the country's real name.
    """
    # Read the countries CSV file to create a mapping from id to name_1
    logger.info(f"Reading countries CSV file: {countries_csv}")
    countries_df = pd.read_csv(countries_csv)
    id_to_name = dict(zip(countries_df['id'], countries_df['name_1']))
    
    # Find all Parquet files in the base directory that match the pattern "country_a_id=*.parquet"
    parquet_files = glob.glob(os.path.join(base_dir, "country_a_id=*.parquet"))
    
    for file_path in parquet_files:
        # Extract the country_a_id from the filename
        filename = os.path.basename(file_path)
        country_id_str = filename.split('=')[1].split('.')[0]
        try:
            country_id = int(country_id_str)
        except ValueError:
            logger.warning(f"Could not extract country ID from filename: {filename}")
            continue
        
        # Get the corresponding country name
        country_name = id_to_name.get(country_id)
        if not country_name:
            logger.warning(f"No country name found for ID {country_id}")
            continue
        
        # Create the new filename
        new_filename = f"{country_name}.parquet"
        new_file_path = os.path.join(base_dir, new_filename)
        
        # Rename the file
        try:
            shutil.move(file_path, new_file_path)
            logger.info(f"Renamed {filename} to {new_filename}")
        except Exception as e:
            logger.error(f"Failed to rename {filename}: {str(e)}")


base_dir = r"/workspace/qgi\qgi-4.0\data\pattern_maker_output"
countries_csv = r"/workspace/qgi\qgi-4.0\data\countries.csv"
rename_parquets_by_country_name(base_dir, countries_csv)
logger.info("Renaming completed")

2025-04-15 16:02:14,263 - Reading countries CSV file: /workspace/qgi\qgi-4.0\data\countries.csv
2025-04-15 16:02:14,453 - Renamed country_a_id=1.parquet to Afghanistan.parquet
2025-04-15 16:02:14,457 - Renamed country_a_id=10.parquet to Argentina.parquet
2025-04-15 16:02:14,460 - Renamed country_a_id=100.parquet to India.parquet
2025-04-15 16:02:14,462 - Renamed country_a_id=101.parquet to Indonesia.parquet
2025-04-15 16:02:14,463 - Renamed country_a_id=102.parquet to Iran.parquet
2025-04-15 16:02:14,467 - Renamed country_a_id=103.parquet to Iraq.parquet
2025-04-15 16:02:14,469 - Renamed country_a_id=104.parquet to Ireland.parquet
2025-04-15 16:02:14,473 - Renamed country_a_id=105.parquet to Isle of Man.parquet
2025-04-15 16:02:14,475 - Renamed country_a_id=106.parquet to Israel.parquet
2025-04-15 16:02:14,478 - Renamed country_a_id=107.parquet to Italy.parquet
2025-04-15 16:02:14,479 - Renamed country_a_id=108.parquet to Ivory Coast.parquet
2025-04-15 16:02:14,483 - Renamed country_a_